
**Getting Started with HuggingFace Models**
1. Load a HuggingFace model (example: google/flan-t5-base, mistralai/Mistral-7B-Instruct).
2. Generate a response for a simple prompt.
3. Observe output quality.


In [9]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [10]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace


In [13]:
# Instruct models need ChatHuggingFace + task="conversational"
# mistralai/Mistral-7B-Instruct-v0.2 is not on providers enabled for this HF token;
# openai/gpt-oss-20b works on the free Inference API router.
endpoint = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-20b",
    temperature=0.7,
    max_new_tokens=3000,
    task="conversational",
)

llm = ChatHuggingFace(llm=endpoint)


In [14]:
res = llm.invoke("What is LangChain?")
print(res.content)


### LangChain in a nutshell

**LangChain** is an open‑source framework that makes it easier to build *applications* that use large language models (LLMs) such as GPT‑4, Claude, Llama, etc.  
Instead of writing raw prompt‑engineering code from scratch, LangChain gives you a set of reusable building blocks (chains, agents, memory, tools, retrievers, etc.) that you can stitch together to create sophisticated, stateful, and context‑aware LLM‑powered apps.

---

## 1. Why LangChain?

| Problem | Traditional approach | LangChain solution |
|---------|---------------------|--------------------|
| Re‑use of prompt logic | Hard‑coded strings, copy‑paste | `PromptTemplate` objects that can be parameterized |
| Managing context over multiple turns | Manual string concatenation | `ConversationBufferMemory`, `VectorStoreRetriever`, etc. |
| Integrating external APIs or tools | Custom wrapper code | `Tool` objects that the agent can call |
| Orchestrating multiple LLM calls | Manual sequencing | `Se

**HuggingFace with LangChain**
1. Integrate the HuggingFace model using LangChain wrappers.
2. Replace OpenAI with HuggingFace in an LLM chain.
3. Test with multiple prompts.

In [15]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.prompts import ChatPromptTemplate

In [25]:
endpoint = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-20b",
    temperature=0.7,
    max_new_tokens=2356,
    task="conversational",
)
llm = ChatHuggingFace(llm=endpoint)

In [26]:
prompt = ChatPromptTemplate.from_template(template="""
    you are a helpful assistant that can answer questions.
    {input}
"""
)

chain = prompt | llm

In [27]:
res = chain.invoke({"input": "What is LangChain?"})
print(res.content)


**LangChain** is an open‑source framework that lets you build, test, and deploy applications powered by large language models (LLMs) – not a model itself, but a toolkit for orchestrating them.  
Think of it as a “pipeline builder” for LLM‑based software, with a rich ecosystem of reusable components.

---

## Core ideas

| Concept | What it is | Why it matters |
|---------|------------|----------------|
| **LLM** | The underlying language model (OpenAI GPT‑4, Anthropic Claude, Cohere, Hugging Face, etc.) | The “brain” that generates text |
| **PromptTemplate** | A reusable prompt with placeholders | Enables consistent, parameterized prompt engineering |
| **Tool** | Any function or API you can call (e.g., a calculator, a database query, a web scraper) | Lets the LLM “act” on the world |
| **Chain** | A sequence of calls (LLM → Tool → LLM → …) | Builds logical workflows (e.g., fetch data → summarize → answer) |
| **Agent** | A higher‑level chain that decides which tool to use based on th


**Chat Prompt Template with HuggingFace**
1. Create a ChatPromptTemplate.
2. Use system + human messages.
3. Generate responses using HuggingFace-backed LLM.
4. Compare with normal prompt template.

In [28]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that can answer questions."),
    ("human", "{input}"),
])
chain = prompt | llm

In [29]:
print(chain.invoke({"input": "What is LangChain?"}))

content='**LangChain** is an open‑source framework that makes it easier to build applications powered by large language models (LLMs) such as GPT‑4, Claude, Llama, etc.  \nIt abstracts away many of the repetitive tasks that come with working with LLMs—prompt formatting, chaining multiple calls, adding memory, integrating external tools, and more—so developers can focus on the business logic of their app.\n\n---\n\n## Core Ideas\n\n| Concept | What it does | Typical use‑case |\n|---------|--------------|------------------|\n| **LLM** | The underlying model (OpenAI, Anthropic, Cohere, local LLMs, etc.) | Text generation, summarization, question answering |\n| **Prompt** | A template that defines how to ask the model | “Translate this sentence to French” |\n| **Chain** | A sequence of steps that may call an LLM, a tool, or another chain | “Summarize article → ask follow‑up question” |\n| **Agent** | A higher‑level component that decides *which* tool or chain to use based on the user’s que

In [31]:
# Compare: plain from_template vs system+human ChatPromptTemplate
plain_prompt = ChatPromptTemplate.from_template(
    "You are a helpful assistant that can answer questions.\n{input}"
)
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that can answer questions. Keep answers short."),
    ("human", "{input}"),
])

question = "What is LangChain in one sentence?"

plain_chain = plain_prompt | llm
chat_chain = chat_prompt | llm

plain_res = plain_chain.invoke({"input": question})
chat_res = chat_chain.invoke({"input": question})

print("=== plain from_template (instructions mixed into one user blob) ===")
print(plain_res.content)
print()
print("=== system + human messages (roles separated) ===")
print(chat_res.content)


=== plain from_template (instructions mixed into one user blob) ===
LangChain is an open‑source framework that simplifies building, testing, and deploying applications that combine large language models with external data sources, memory, and tools.

=== system + human messages (roles separated) ===
LangChain is an open‑source framework that lets developers build applications by chaining together language‑model calls, prompts, and external data or tools.


Takeaway: ChatPromptTemplate.from_messages maps cleanly to chat models 
(system role stays system). from_template is fine for simple strings, 
but mixes instructions into the human turn.